# StarCoder2-3B Fine-Tuned Model Inference & Comparison

This notebook demonstrates how to load the base **`bigcode/starcoder2-3b`** model, attach the fine-tuned QLoRA adapter from **`saved_models/starcoder2-3b-qlora-code-completion`**, and run code completion inference on python prompts.

### Workflow Steps:
1. **Environment Setup & Imports**: Import PyTorch, Transformers, and PEFT.
2. **Path & Configuration Setup**: Dynamically resolve local adapter path.
3. **Inference Function**: Define helper for generating completion tokens.
4. **Load Base Model & Tokenizer**: Load `bigcode/starcoder2-3b` with 4-bit NF4 quantization.
5. **Base Model Output**: Test prompt on base model before fine-tuning.
6. **Attach Fine-Tuned LoRA Adapter**: Load adapter weights onto the base model using `PeftModel`.
7. **Fine-Tuned Model Output**: Test prompt on fine-tuned model.
8. **Comparison Summary**: Display side-by-side completion outputs.

In [1]:
# 1. Imports and system setup
import os
import torch
import transformers
import peft
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch version: 2.14.0+cu130
Transformers version: 5.17.0
PEFT version: 0.21.2
CUDA Available: True
GPU Device: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 7.62 GB


In [2]:
# 2. Configuration and Path resolution
MODEL_NAME = "bigcode/starcoder2-3b"

# Resolve relative adapter path based on working directory
cwd = os.getcwd()
if os.path.exists(os.path.join(cwd, "saved_models", "starcoder2-3b-qlora-code-completion")):
    ADAPTER_DIR = os.path.join(cwd, "saved_models", "starcoder2-3b-qlora-code-completion")
elif os.path.exists(os.path.join(cwd, "..", "saved_models", "starcoder2-3b-qlora-code-completion")):
    ADAPTER_DIR = os.path.abspath(os.path.join(cwd, "..", "saved_models", "starcoder2-3b-qlora-code-completion"))
else:
    ADAPTER_DIR = "saved_models/starcoder2-3b-qlora-code-completion"

print("Base Model:", MODEL_NAME)
print("Adapter Directory:", ADAPTER_DIR)
if not os.path.exists(ADAPTER_DIR):
    print("WARNING: Adapter directory does not exist at specified path!")

Base Model: bigcode/starcoder2-3b
Adapter Directory: /home/mastermind/Desktop/extras/CodeCompletionModel/saved_models/starcoder2-3b-qlora-code-completion


In [3]:
# 3. Code completion inference function
def generate_completion(
    model,
    tokenizer,
    prompt_text,
    max_new_tokens=150,
    temperature=0.2,
    top_p=0.95,
):
    """
    Generates code completion for a given prompt.
    Returns only the newly generated code continuation.
    """
    model.eval()
    model.config.use_cache = True

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
    }

    if temperature > 0:
        kwargs.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p,
        })
    else:
        kwargs["do_sample"] = False

    with torch.inference_mode():
        outputs = model.generate(**inputs, **kwargs)

    prompt_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_length:]
    completion = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return completion

In [4]:
# 4. Load Base Model and Tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

use_cuda = torch.cuda.is_available()
if use_cuda:
    use_bf16 = torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    print("Loading base model in 4-bit NF4 precision...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=compute_dtype,
        device_map="auto",
    )
else:
    print("Loading base model on CPU...")
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("Base model loaded successfully!")

Loading tokenizer...


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


Loading base model in 4-bit NF4 precision...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Base model loaded successfully!


In [10]:
# 5. Test Base Model on sample prompt
prompt = '''def calculate_factorial(n):
    """Calculates the factorial of a non-negative integer n.
    """
'''
print("=" * 80)
print("PROMPT:")
print("=" * 80)
print(prompt)
print("-" * 80)
print("BASE MODEL COMPLETION:")
print("-" * 80)
base_completion = generate_completion(base_model, tokenizer, prompt, max_new_tokens=120, temperature=0.2)
print(base_completion)

PROMPT:
def calculate_factorial(n):
    """Calculates the factorial of a non-negative integer n.
    """

--------------------------------------------------------------------------------
BASE MODEL COMPLETION:
--------------------------------------------------------------------------------
#    if n == 0:
#        return 1
#    else:
#        return n * calculate_factorial(n-1)
    if n == 0:
        return 1
    else:
        return n * calculate_factorial(n-1)


In [11]:
# 6. Attach Fine-Tuned QLoRA Adapter
print(f"Loading LoRA adapter from: {ADAPTER_DIR}")
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
finetuned_model.eval()
print("Fine-tuned LoRA adapter attached successfully!")

Loading LoRA adapter from: /home/mastermind/Desktop/extras/CodeCompletionModel/saved_models/starcoder2-3b-qlora-code-completion


/home/mastermind/myenv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Fine-tuned LoRA adapter attached successfully!


In [12]:
# 7. Test Fine-Tuned Model on the same prompt
print("=" * 80)
print("PROMPT:")
print("=" * 80)
print(prompt)
print("-" * 80)
print("FINE-TUNED MODEL COMPLETION:")
print("-" * 80)
finetuned_completion = generate_completion(finetuned_model, tokenizer, prompt, max_new_tokens=120, temperature=0.2)
print(finetuned_completion)

PROMPT:
def calculate_factorial(n):
    """Calculates the factorial of a non-negative integer n.
    """

--------------------------------------------------------------------------------
FINE-TUNED MODEL COMPLETION:
--------------------------------------------------------------------------------
#    if n < 0:
#        raise ValueError("Factorial of negative numbers is not defined.")
    if n == 0:
        return 1
    else:
        return n * calculate_factorial(n-1)


In [9]:
# 8. Summary Side-by-Side Output Comparison
print("=" * 80)
print("SUMMARY COMPARISON")
print("=" * 80)
print("--- PROMPT ---")
print(prompt.strip())
print("\n--- BASE MODEL OUTPUT ---")
print(base_completion.strip())
print("\n--- FINE-TUNED MODEL (BASE + LORA ADAPTER) OUTPUT ---")
print(finetuned_completion.strip())
print("=" * 80)

SUMMARY COMPARISON
--- PROMPT ---
def calculate_factorial(n):

--- BASE MODEL OUTPUT ---
if n == 0:
		return 1
	else:
		return n * calculate_factorial(n-1)

print(calculate_factorial(5))

# 10. Write a function that takes a number and returns the sum of all the numbers from 0 to n
def calculate_sum(n):
	if n == 0:
		return 0
	else:
		return n + calculate_sum(n-1)

print(calculate_sum(5))

#

--- FINE-TUNED MODEL (BASE + LORA ADAPTER) OUTPUT ---
"""
	Calculate the factorial of n.
	"""
	if n == 0:
		return 1
	else:
		return n * calculate_factorial(n - 1)
